# LINDAS Tutorial

<img src="https://lindas.admin.ch/static-assets/img/lindaslogo.png" style="width:25%; float:right; padding:20px">

Dieses Tutorial ist dazu gedacht, eine Einführung in das Linked Data Ökosystem LINDAS der schweizerischen Eigenossenschaft zu geben.

## Lernziele

- Sie verstehen in Grundzügen, was Linked Data ist
- Sie kennen das Datenformat RDF
- Sie kennen URI als Identifier für Objekte
- Sie wissen, was SPARQL Queries sind
- Sie verstehen, was LINDAS ist
- Sie können Daten in LINDAS finden
- Sie haben einen Überblick über die verfügbaren Daten in LINDAS
- Sie können SPARQL Queries auf LINDAS ausführen

## Interaktives Notebook

<img src="https://jupyter.org/assets/homepage/main-logo.svg" style="width:15%; float:left; padding:20px">

Dieses Tutorial ist ein sogenanntes **interaktives JupyterLite Notebook**. In diesem Notebook können Sie den Inhalt der einzelnen Zellen interaktiv ändern und diese Zellen direkt ausführen, um das Ergebnis Ihrer Änderungen sofort zu sehen. Die Zellen enthalten entweder [Markdown](https://en.wikipedia.org/wiki/Markdown)-Inhalt (wie diese Zelle) oder ausführbaren Python-Quellcode. Dies ist für ein Tutorial sehr hilfreich, weil Inhalte beliebig mit ausführbarem Quellcode kombiniert werden können. Es können also Abfragen gezeigt werden, diese erklärt werden und darauf weiter aufgebaut werden.

- **Um direkt loslegen zu können klicken Sie oben im Menu auf Run -> Run All Cells.**  
- **Einzelne ausgewählte Zellen können sie danach mit dem "Play-Button" erneut ausführen und so Abfragen individuell anpassen.**

Das Notebook startet mit einem [Setup](#Setup) der Programierumgebung. Das eigentliche Tutorial startet [hier](#Linked-Data-Einführung).

*Zusätzliche Informationen zu JupyterLite:*  
JupyterLite is ein spezielles Jupyter Notebook mit dem Vorteil, vollständig browserbasiert zu sein, ohne eine aufwändige Backend-Infrastruktur zu benötigen. Der Nachteil ist, dass die erstmalige Ausführung der Zellen einige Zeit in Anspruch nehmen kann, weil eine erhebliche Menge von Daten geladen werden muss. Nachfolgende Ausführungen werden aufgrund der gespeicherten Daten in Ihrem Browser-Cache viel schneller sein.

Wenn Sie mehr über die Handhabung von Jupyter Notebooks wissen wollen, finden Sie hier zwei nützliche Ressourcen:

- [Die JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [Das Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)


## Setup

Eine SPARQL Abfrage ist nichts anderes als ein POST-Request an den entsprechenden Triple Store Datenbank Server. Um diese Requests und die erhaltenen Antworten einfacher handhaben zu können, enthält dieses Notebook vorbereiteten Python Code, der nachfolgend importiert wird. Zusätzlich werden verschiedene Module installiert resp. importiert - u.a. wird das `pandas` Modul importiert, welches die Möglichkeit bietet, die tabellarischen Daten der SPARQL Abfragen als [Pandas Dataframes](https://pandas.pydata.org/docs/index.html) weiter zu verarbeiten. 

In [1]:
# Installation von folium, um Daten auf Landkarten darzustellen
%pip install -q folium

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import branca
from ext.sparql import query, display_result

# Linked Data Einführung

Linked Data beschreibt ein **Framework für den Umgang mit Daten**, die sowohl für Menschen nützlich sein sollen, als auch maschinenlesbar sind inkl. einer von Computern verarbeitbaren Semantik. Also sowohl Menschen als auch Computer sollen die Daten "verstehen" und interpretieren können. 

## RDF

Das für Linked Data verwendete Datenformat ist RDF (Resource Description Framework). Das bedeutet, dass die Daten nicht als Tabellen (wie beispielsweise in relationalen Datenbanken) sondern als **Triples** abgespeichert werden. Triples folgen der grammatikalischen Struktur **Subjekt -> Prädikat -> Objekt** und können auch als grammatikalischer Satz verstanden werden. 

Die Information "**Der Apfel ist grün**" wird also mit dem Tripel **Apfel -> ist -> grün** ausgedrückt. Alle Teile eines Triples sind dabei durch weitere Eigenschaften definiert und beschrieben die wiederum in Form von Triples beschrieben sind. Diese vielseitigen Verknüpfungen führen zu einer Netzwerkstruktur, zu einem sogenannten Graphen.

Nachfolgendes Bild aus dem [W3C Primer für RDF](https://www.w3.org/TR/rdf11-primer/) veranschaulicht diese Zusammenhänge:

![](https://www.w3.org/TR/rdf11-primer/example-graph.jpg)

## URI

Eine weitere wichtige Eigenschaft von Linked Data ist, dass alle Teile eines Triples, also Subjekt, Prädikat und Objekt weltweit eindeutig identifizierbar sind. Dafür werden URI (Universal Resource Identifier) eingesetzt. Die URI https://ld.admin.ch/municipality/351 beispielsweise ist der weltweit eindeutige Identifier für die Stadt Bern (natürlich existieren an anderen Orten noch weitere Identifier für die Stadt Bern). Typischerweise lassen sich URI **dereferenzieren**, das heisst, ein Request auf die entsprechende URI (bspw. in dem man sie in die Adresszeile eines Browsers eintippt) führt zu einer Website, die Infos zur entsprechenden URI enthält. Im Falle der URI der Stadt Bern wird man auf eine Webpage weitergeleitet, die diverse Informationen zur Stadt Bern enthält. Diese Informationen sind exakt diejenigen Triples in LINDAS, deren Subjekt die Stadt Bern ist.

## SPARQL

<img src="https://cygri.github.io/rdf-logos/svg/sparql.svg" style="width:20%; float:right; padding:20px">

SPARQL ist eine Query-Sprache für Linked Data Triple Stores. Für eine allgemeine Einführung in SPARQL siehe z.B.: https://en.wikibooks.org/wiki/SPARQL. 

Abfragen (engl. Queries) können entweder direkt über das [SPARQL-Interface von LINDAS](https://ld.admin.ch/sparql) eingegeben und ausgeführt werden, oder als HTTP-POST Request an den SPARQL-Endpoint (https://ld.admin.ch/query) gesendet werden.

Die letztere Methode erlaubt es, eigene Anwendungen zu bauen, die automatisch aktuelle Daten von LINDAS abfragen können. Für dieses Tutorial verwenden wir diese Methode. Die Abfragen sind jedoch in beiden Fällen identisch. Für die Abfrage über das Interface kopieren Sie einfach den Teil zwischen den `"""` und fügen sie im SPARQL-Interface ein.

### Pattern Matching

SPARQL Queries sind Aufträge an den Computer, bestimme Muster (Pattern) in den Daten zu finden (matching). Es können also mit Hilfe von SPARQL Muster vorgegeben werden, und die Datenbank gibt alle Triples zurück, die dieses Muster erfüllen. Einzelne Positionen der Triples können dabei bei einer Abfrage bewusst undefiniert gelassen und mit einer Variable bezeichnet werden. Variablen beginnen mit `?` und werden bei der Abfrage durch alle möglichen Elemente gefüllt, die dieses Pattern erfüllen. Eine ausführlichere Anleitung zum Pattern Matching ist [hier](https://programminghistorian.org/en/lessons/retired/graph-databases-and-SPARQL#rdf-in-brief) zu finden. 

## Weitere Informationen zu Linked Data

Wer vertieft in das Thema Linked Data einsteigen möchte, dem sei beispielsweise [diese Youtube Playlist](https://www.youtube.com/watch?v=ON0wf0SEPx8&list=PLoOmvuyo5UAfY6jb46jCpMoqb-dbVewxg) empfohlen.

# Datasets

Daten in LINDAS sind in Datasets gruppiert. Diese fassen gleichartige Daten zusammen. Datasets in LINDAS haben den [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [schema:Dataset](https://schema.org/Dataset).

## Alle Datasets

Die erste SPARQL Query dient dazu, herauszufinden, welche Datasets in LINDAS aktuell existieren:

In [3]:
df = await query("""

SELECT * WHERE {
    
    # suche alle Subjekte, der Prädikate und Objekte die entsprechenden Werte haben
    ?dataset <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://schema.org/Dataset>.

} LIMIT 10 

""")

display_result(df)

,dataset
0,https://culture.ld.admin.ch/.well-known/dataset/isil
1,https://register.ld.admin.ch/.well-known/dataset/opendataswiss-meta
2,https://energy.ld.admin.ch/elcom/electricityprice
3,https://energy.ld.admin.ch/elcom/electricityprice-canton
4,https://energy.ld.admin.ch/elcom/electricityprice-swiss
5,https://lod.opentransportdata.swiss/.well-known/dataset/didok
6,https://lod.opentransportdata.swiss/.well-known/dataset/nova
7,https://culture.ld.admin.ch/sfa/StateAccounts_Office/1/
8,https://culture.ld.admin.ch/sfa/StateAccounts_Domain/1/
9,https://environment.ld.admin.ch/foen/ubd0037/1/


Diese URI der Datasets können angeklickt werden und damit öffnet sich eine Website mit weiteren Informationen zum entsprechenden Dataset.

Um, die SPARQL Queries übersichtlicher zu gestalten, können häufig gebrauchte URI mit Prefixes abgekürzt werden. Ausserdem existiert die generische Abkürzung 'a' für http://www.w3.org/1999/02/22-rdf-syntax-ns#type. Damit wird die oben gezeigte Query zu:

In [4]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT * WHERE {
    
    # suche alle Subjekte, der Prädikate und Objekte die entsprechenden Werte haben
    ?dataset a schema:Dataset.

} LIMIT 10 

""")

display_result(df)

,dataset
0,https://culture.ld.admin.ch/.well-known/dataset/isil
1,https://register.ld.admin.ch/.well-known/dataset/opendataswiss-meta
2,https://energy.ld.admin.ch/elcom/electricityprice
3,https://energy.ld.admin.ch/elcom/electricityprice-canton
4,https://energy.ld.admin.ch/elcom/electricityprice-swiss
5,https://lod.opentransportdata.swiss/.well-known/dataset/didok
6,https://lod.opentransportdata.swiss/.well-known/dataset/nova
7,https://culture.ld.admin.ch/sfa/StateAccounts_Office/1/
8,https://culture.ld.admin.ch/sfa/StateAccounts_Domain/1/
9,https://environment.ld.admin.ch/foen/ubd0037/1/


Um etwas mehr, als nur die URI der Datasets als Resultat zu erhalten, kann gleichzeitig auch nach dem Namen des Datasets gefragt werden. Da die Namen in verschiedenen Sprachen angegeben sind, muss nach der gewünschten Sprache gefiltert werden, ansonsten würde man jedes Dataset in jeder angegebenen Sprache zurückerhalten:

In [5]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name.
        
    FILTER(lang(?name) = 'de')
}
ORDER BY (?dataset)

""")

display_result(df)

,dataset,name
0,https://agriculture.ld.admin.ch/fsvo/animal-disease/dataset,Tierseuchen
1,https://culture.ld.admin.ch/.well-known/dataset/ais,Archivdatenbank des Schweizerischen Bundesarchiv
2,https://culture.ld.admin.ch/.well-known/dataset/isil,ISIL-Kennzeichen
3,https://culture.ld.admin.ch/sfa/StateAccounts_Category/1/,Staatsrechnungen - Kategorie
4,https://culture.ld.admin.ch/sfa/StateAccounts_Category/2/,Staatsrechnungen - Kategorie
5,https://culture.ld.admin.ch/sfa/StateAccounts_Category/3/,Staatsrechnungen - Kategorie
6,https://culture.ld.admin.ch/sfa/StateAccounts_Category/4/,Staatsrechnungen - Kategorie
7,https://culture.ld.admin.ch/sfa/StateAccounts_Category/5/,Staatsrechnungen - Kategorie
8,https://culture.ld.admin.ch/sfa/StateAccounts_Category/6,Staatsrechnungen - Kategorie
9,https://culture.ld.admin.ch/sfa/StateAccounts_Domain/1/,Staatsrechnungen - Bereich


Es zeigt sich, dass diverse Datasets in verschiedenen Versionen existieren. Dasjenige mit der grössten Versionsnummer das aktuellste. Das bedeutet nicht unbedingt, dass es sich um aktuellere Daten handelt, eher noch, dass die Metadaten in besserer und vollständigeren Qualität vorliegen.

Ein Klick bspw. auf https://environment.ld.admin.ch/foen/ubd000504/3 öffnet die entsprechende Website mit weiteren Informationen zum Dataset, bspw. ist dort angegeben, dass der [purl:creator](http://purl.org/dc/terms/creator) des Datensatzes folgendende URI ist: https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu. Es handelt sich also um einen Datensatz des Bundesamts für Umwelt.

## Alle Datenlieferanten

Die Datenlieferanten der Datasets sind über [purl:creator](http://purl.org/dc/terms/creator) angehängt. Folgende SPARQL Query liefert alle Datenlieferanten in LINDAS:

In [6]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT DISTINCT ?owner ?name WHERE {
    ?dataset a schema:Dataset;
        purl:creator ?owner.
    
    OPTIONAL {
    ?owner schema:name ?name.
    
    FILTER(lang(?name) = "de")
    }
}

""")

display_result(df)

,owner,name
0,https://register.ld.admin.ch/opendataswiss/org/schweizerisches-bundesarchiv-bar,Schweizerisches Bundesarchiv BAR
1,https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-statistik-bfs,Bundesamt für Statistik BFS
2,https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu,Bundesamt für Umwelt BAFU
3,https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-energie-bfe,Bundesamt für Energie BFE
4,https://register.ld.admin.ch/staatskalender/organization/10002463,Eidgenössisches Amt für das Handelsregister
5,https://register.ld.admin.ch/staatskalender/organization/20030954,Eidgenössische Elektrizitätskommission
6,https://ld.admin.ch/FCh,Bundeskanzlei
7,https://register.ld.admin.ch/opendataswiss/org/efv_finanzstatistik,Finanzstatistik EFV
8,https://register.ld.admin.ch/opendataswiss/org/elcom,


## Datasets eines spezifischen Datenlieferanten

Das Prädikat [purl:creator](http://purl.org/dc/terms/creator) kann benutzt werden, um nur Daten des BAFU anzuzeigen:

In [7]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name;
        purl:creator <https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu>.
        
    FILTER(lang(?name) = 'de')
}

""")

display_result(df)

,dataset,name
0,https://environment.ld.admin.ch/foen/ubd0037/1/,Lärmbelastung durch Verkehr
1,https://environment.ld.admin.ch/foen/ubd0104/1/,Qualität der Badegewässer
2,https://environment.ld.admin.ch/foen/ubd0066/1/,Schwermetallbelastung des Bodens
3,https://environment.ld.admin.ch/foen/ubd0066/2/,Schwermetallbelastung des Bodens
4,https://environment.ld.admin.ch/foen/ubd0037/2/,Lärmbelastung durch Verkehr
5,https://environment.ld.admin.ch/foen/ubd0066/3/,Schwermetallbelastung des Bodens
6,https://environment.ld.admin.ch/foen/ubd0104/2/,Qualität der Badegewässer
7,https://environment.ld.admin.ch/foen/ubd0037/3/,Lärmbelastung durch Verkehr
8,https://environment.ld.admin.ch/foen/ubd0066/4/,Schwermetallbelastung des Bodens
9,https://environment.ld.admin.ch/foen/ubd0104/3/,Qualität der Badegewässer


# Spezifische Datasets

In den nachfolgenden Abschnitten werden nun bestimmte Datasets näher untersucht und dabei wird die Struktur der eigentlichen Daten innerhalb der Datasets weiter erläutert. Zusätzlich wird aufgezeigt, wie georeferenzierte Daten direkt im Jupyter Notebook mit Hilfe einer Kartendarstellung angezeigt werden können.

## Qualität der Badegewässer

### Aktuellste Version des Datasets

Die erste Aufgabe ist, die aktuellste Version des entsprechenden Datasets herauszufinden. Ein Klick auf eine beliebige Version des Datensatzes (bspw. https://environment.ld.admin.ch/foen/ubd0104/3/) zeigt, dass dieser Datensatz und alle anderen einen gemeinsamen [purl:identifier](http://purl.org/dc/terms/identifier) "ubd0104" aufweisen. Somit kann folgende SPARQL Query mit einer [SPARQL subquery](https://en.wikibooks.org/wiki/SPARQL/Subqueries) Konstruktion das aktuellste Dataset bestimmen:

In [8]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT * WHERE {

    ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?maxversion.
    {
        SELECT (max(?version) as ?maxversion) WHERE {
            ?dataset a schema:Dataset;
                purl:identifier "ubd0104";
                schema:version ?version.
        }
    }
}

""")

display_result(df)

,dataset,maxversion
0,https://environment.ld.admin.ch/foen/ubd0104/6,6


### ObservationSet und Observation

Weil dieses Dataset vom [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [cube:Cube](https://cube.link/#Cube) ist, sind die eigentlichen Messwerte innerhalb eines [cube:ObservationSet](https://cube.link/#ObservationSet) zusammengefasst:

In [9]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT ?observation WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

LIMIT 10

""")

display_result(df)

,observation
0,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-05-23/E.coli/CH10001
1,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-05-26/E.coli/CH10001
2,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-05-23/Enterokokken/CH10001
3,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-08-13/Enterokokken/CH10001
4,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-08-11/E.coli/CH10001
5,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-07-24/E.coli/CH10001
6,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2018-09-03/E.coli/CH10001
7,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-06-23/Enterokokken/CH10001
8,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-05-26/Enterokokken/CH10001
9,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-08-11/Enterokokken/CH10001


### Messwerttabelle

Um das verwendete Vokabular besser zu verstehen, kann eine einzelne Observation angeklickt werden und damit folgende Query erstellt werden:

In [10]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>

SELECT ?name ?station ?dateOfProbing ?parameterType ?value WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    ?observation ubd0104:dateofprobing ?dateOfProbing;
        ubd0104:station ?station;
        ubd0104:parametertype ?parameterType;
        ubd0104:station ?station;
        ubd0104:value ?value.
    
    ?station schema:name ?name.
        
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

LIMIT 100

""")

display_result(df)

,name,station,dateOfProbing,parameterType,value
0,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2019-05-23,E.coli,210
1,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-05-26,E.coli,80
2,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2019-05-23,Enterokokken,62
3,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2019-08-13,Enterokokken,31
4,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-08-11,E.coli,21
5,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2019-07-24,E.coli,17
6,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2018-09-03,E.coli,15
7,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-06-23,Enterokokken,14
8,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-05-26,Enterokokken,14
9,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-08-11,Enterokokken,12


### Kartendarstellung der Messstationen

Ein nächstes Ziel könnte sein, alle verfügbaren Messstationen auf einer Landkarte darzustellen. Die dazu notwendige SPARQL Query ist die folgende:

In [11]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>


SELECT DISTINCT ?station ?canton ?name ?latitude ?longitude WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    ?observation ubd0104:station ?station.
    ?station ubd0104:kanton ?canton;
        schema:name ?name;
        schema:latitude ?latitude;
        schema:longitude ?longitude.
        
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

""")

display_result(df)

,station,canton,name,latitude,longitude
0,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,FR,Plage de Gumefens,46.6753,7.0879
1,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10004,FR,Gemeinde Strandbad,46.9279,7.1109
2,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10007,FR,Plage de Portalban,46.9227,6.9534
3,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10009,FR,Nouvelle plage,46.8571,6.8483
4,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1013,ZH,Strandbad Maur,47.3456,8.6747
5,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1016,ZH,Strandbad Uster,47.3414,8.6911
6,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1029,ZH,Strandbad Baumen,47.3583,8.7855
7,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1050,ZH,Strandbad Küsnacht,47.3093,8.5826
8,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1058,ZH,Seebad Lattenberg,47.2390,8.7171
9,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1061,ZH,Strandbad Bürger,47.2891,8.5749


Um die Kartendarstellung zu erzeugen, nutzen wir das Python Modul [folium](https://python-visualization.github.io/folium/), welches wir zuerst noch importieren müssen:

In [12]:
# Karte erstellen und um das Mittel der Koordinaten zentrieren
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], 
               tiles="https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.pixelkarte-farbe/default/current/3857/{z}/{x}/{y}.jpeg",
               attr='<a href="https://www.swisstopo.admin.ch" target=_blank>©swisstopo</a>',
               zoom_start=8)

# Hinzufügen der Marker
for i in range(0,len(df)):
    folium.Marker(
        location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
        popup=folium.Popup(folium.Html('<a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a>', script=True), max_width=500),
        icon=folium.Icon(icon='person-swimming', prefix='fa')
   ).add_to(m)

#Karte anzeigen
m

### Zusätzliche Informationen in Karte

Um die Karte zu verbessern, wollen wir die Markierung in verschiedenen Farben anzeigen, die der Qualität der letzten Messungen entsprechen, und wenn man auf die Markierung klickt, sollte ein Popup mit zusätzlichen Informationen erscheinen.

In [13]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>

SELECT ?name ?station ?latitude ?longitude ?date ?value WHERE {

    ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version 6;
            cube:observationSet/cube:observation ?observation.

        ?observation ubd0104:station ?station;
            ubd0104:parametertype "E.coli";
            ubd0104:dateofprobing ?date;
            ubd0104:value ?value.
        
        ?station schema:name ?name;
            schema:latitude ?latitude;
            schema:longitude ?longitude.
    

    {
    SELECT ?station (max(?date) as ?date) WHERE {

        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version 6;
            cube:observationSet/cube:observation ?observation.

        ?observation ubd0104:station ?station;
            ubd0104:parametertype "E.coli";
            ubd0104:dateofprobing ?date.
    }

    GROUP BY ?station
    }
}
""")

display_result(df)

,name,station,latitude,longitude,date,value
0,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,46.6753,7.0879,2021-09-09,2
1,Gemeinde Strandbad,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10004,46.9279,7.1109,2021-09-07,11
2,Plage de Portalban,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10007,46.9227,6.9534,2021-08-26,1
3,Nouvelle plage,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10009,46.8571,6.8483,2021-08-12,18
4,Strandbad Maur,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1013,47.3456,8.6747,2021-08-19,41
5,Strandbad Uster,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1016,47.3414,8.6911,2021-08-19,41
6,Strandbad Baumen,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1029,47.3583,8.7855,2021-08-19,135
7,Strandbad Küsnacht,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1050,47.3093,8.5826,2021-08-11,1
8,Seebad Lattenberg,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1058,47.2390,8.7171,2021-08-11,3
9,Strandbad Bürger,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1061,47.2891,8.5749,2021-08-11,1


Jetzt soll die Karte erstellt werden:

In [14]:
# Mapping von Qualität zu Farbe erstellen
conditions = [(df['value'] < 10),
              (df['value'] >= 10) & (df['value'] < 50),
              (df['value'] >= 50) & (df['value'] < 100),
              (df['value'] >= 100)
             ]

values = ["green", "darkgreen", "orange", "red"]

df['quality'] = np.select(conditions, values)

# Karte erstellen und um das Mittel der Koordinaten zentrieren
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], tiles="stamenterrain", zoom_start=8)

# Hinzufügen der Marker
for i in range(0,len(df)):
    folium.Marker(
        location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
        popup=folium.Popup(folium.Html('<p><a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a></p>' + 
                                       '<p>Date: ' + df.iloc[i]['date'] + '</p>' + 
                                       '<p>Value E.Coli = ' + str(df.iloc[i]["value"]) + '</p>', script=True), max_width=500),
        icon=folium.Icon(icon='person-swimming', prefix='fa', color = df.iloc[i]['quality'])
   ).add_to(m)

folium.TileLayer('https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.pixelkarte-farbe/default/current/3857/{z}/{x}/{y}.jpeg',
                attr='<a href="https://www.swisstopo.admin.ch" target=_blank>©swisstopo</a>',
                name="swisstopo").add_to(m)
folium.LayerControl().add_to(m)    

# Karte anzeigen
m

## Public Transport

### LINDAS: Haltestellen öffentlicher Verkehr

In [15]:
df = await query("""

SELECT *  
WHERE {
  ?station a <http://vocab.gtfs.org/terms#Station>.
} LIMIT 10

""")

display_result(df)

,station
0,https://lod.opentransportdata.swiss/didok/8503025
1,https://lod.opentransportdata.swiss/didok/8501665
2,https://lod.opentransportdata.swiss/didok/8502108
3,https://lod.opentransportdata.swiss/didok/8517320
4,https://lod.opentransportdata.swiss/didok/8502207
5,https://lod.opentransportdata.swiss/didok/8501664
6,https://lod.opentransportdata.swiss/didok/8589087
7,https://lod.opentransportdata.swiss/didok/8516825
8,https://lod.opentransportdata.swiss/didok/8503504
9,https://lod.opentransportdata.swiss/didok/8573043


### Swisstopo: Bahnhöfe pro Gemeinde

In [16]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT DISTINCT ?muni_name (COUNT(?stop) as ?number_of_stops) WHERE {
  ?stop a <http://vocab.gtfs.org/terms#Stop>;
      <https://geo.ld.admin.ch/def/transportation/meansOfTransportation> <https://geo.ld.admin.ch/vocabulary/MeansOfTransportation/B>;
      <http://schema.org/containedInPlace> ?muni.
    ?muni schema:name ?muni_name.
} GROUP BY ?muni_name ORDER BY DESC(?number_of_stops) LIMIT 20

""", "G")

display_result(df)

,muni_name,number_of_stops
0,Zürich,30
1,Montreux,17
2,Blonay - Saint-Légier,16
3,Ollon,15
4,St. Gallen,13
5,Bern,12
6,Bex,11
7,Köniz,11
8,Winterthur,10
9,Aigle,9


### Swisstopo: Gemeinden ohne ÖV Haltestellen

In [17]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?muni ?muni_name ?geometry WHERE {

    ?muni a <http://schema.org/AdministrativeArea>;
        <http://www.geonames.org/ontology#featureCode> <http://www.geonames.org/ontology#A.ADM3>;
        <http://purl.org/dc/terms/hasVersion> ?version;
        schema:name ?muni_name.
    ?version <http://purl.org/dc/terms/issued> "2023-01-01"^^xsd:date;
        <http://www.opengis.net/ont/geosparql#defaultGeometry>/<http://www.opengis.net/ont/geosparql#asWKT> ?geometry
        
    FILTER NOT EXISTS {
        ?stop a <http://vocab.gtfs.org/terms#Stop>;
            <http://schema.org/containedInPlace> ?muni.
    }

}

""", "G")

display_result(df[["muni", "muni_name"]])

,muni,muni_name
0,https://geo.ld.admin.ch/boundaries/municipality/2061,Auboranges
1,https://geo.ld.admin.ch/boundaries/municipality/2216,Pierrafortscha
2,https://geo.ld.admin.ch/boundaries/municipality/2261,Greng
3,https://geo.ld.admin.ch/boundaries/municipality/2271,Meyriez
4,https://geo.ld.admin.ch/boundaries/municipality/2549,Kammersrohr
5,https://geo.ld.admin.ch/boundaries/municipality/2585,Walterswil (SO)
6,https://geo.ld.admin.ch/boundaries/municipality/3003,Schönengrund
7,https://geo.ld.admin.ch/boundaries/municipality/322,Auswil
8,https://geo.ld.admin.ch/boundaries/municipality/336,Reisiswil
9,https://geo.ld.admin.ch/boundaries/municipality/339,Rohrbachgraben


In [18]:
df = gpd.GeoDataFrame(df)

df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"], crs="EPSG:4326")

sw = df.bounds[["miny", "minx"]].min().values.tolist()
ne = df.bounds[["maxy", "maxx"]].max().values.tolist()

display(df.head())

,muni,muni_name,geometry
0,https://geo.ld.admin.ch/boundaries/municipalit...,Auboranges,"POLYGON ((6.79921 46.58355, 6.79921 46.58363, ..."
1,https://geo.ld.admin.ch/boundaries/municipalit...,Pierrafortscha,"POLYGON ((7.17583 46.79389, 7.17620 46.79354, ..."
2,https://geo.ld.admin.ch/boundaries/municipalit...,Greng,"POLYGON ((7.09938 46.91138, 7.09940 46.91134, ..."
3,https://geo.ld.admin.ch/boundaries/municipalit...,Meyriez,"POLYGON ((7.08830 46.93289, 7.09879 46.93822, ..."
4,https://geo.ld.admin.ch/boundaries/municipalit...,Kammersrohr,"POLYGON ((7.58977 47.25123, 7.58979 47.25141, ..."


In [19]:
geo_json = df.to_json()

m = folium.Map(location=[47, 7], tiles='openstreetmap', zoom_start=10)

folium.GeoJson(geo_json, 
               name="geojson",
               style_function = lambda feature: {
                    "fillOpacity": 0.7,
                    "line_opacity": 0.5,
                    "weight": 1,
                    "fillColor": "blue"
               },
               tooltip=folium.features.GeoJsonTooltip(
                     fields=["muni_name"],
                     style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"),
                     sticky=False
               )
).add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds([sw, ne])
m

### Swisstopo: Veraltete Daten

Folgende ÖV Haltestellen sind aktuell einer nicht mehr existierenden Gemeinde zugeordnet:

In [20]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT ?stop ?stop_name ?numi_name WHERE {

    ?stop a <http://vocab.gtfs.org/terms#Stop>;
        schema:name ?stop_name;
        schema:containedInPlace ?muni.
    ?muni schema:name ?numi_name.
    
     FILTER NOT EXISTS {
        ?muni purl:hasVersion ?version.    
        ?version purl:issued "2023-01-01"^^xsd:date;
    }
}

""", "G")

display_result(df)

,stop,stop_name,numi_name


# Choropleth Karten

Choropleth Karten sind beliebte Mittel, einen Messwert für ein bestimmtes geografisches gebiet zu visualisieren. 

## Bevölkerungsdichte von Schweizer Gemeinden

Nachfolgend sollen Daten zur Bevölkerungsdichte visualisiert werden, die aus der Fläche und Anzahl Einwohner einer Gemeinde berechnet werden:

In [21]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX ogis: <http://www.opengis.net/ont/geosparql#>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?bfs ?name ?version ?population ?area ?geometry WHERE {

?muni gn:featureCode gn:A.ADM3;
    schema:name ?name;
    schema:about ?about;
    <https://geo.ld.admin.ch/def/bfsNumber> ?bfs;
    purl:hasVersion ?version.
    
?version purl:issued "2023-01-01"^^xsd:date;
    ogis:defaultGeometry/ogis:asWKT ?geometry;
    gn:population ?population;
    <http://dbpedia.org/property/area> ?area;
    gn:parentADM1 ?canton.
?canton schema:name "Vaud".
    
}

""", "G")

df = gpd.GeoDataFrame(df)

df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"], crs="EPSG:4326")

df["density"] = df["population"] / df["area"]

sw = df.bounds[["miny", "minx"]].min().values.tolist()
ne = df.bounds[["maxy", "maxx"]].max().values.tolist()

display(df.head())

,bfs,name,version,population,area,geometry,density
0,5401,Aigle,https://geo.ld.admin.ch/boundaries/municipalit...,10776,1641.0,"POLYGON ((6.93227 46.32444, 6.93217 46.32484, ...",6.566728
1,5402,Bex,https://geo.ld.admin.ch/boundaries/municipalit...,8080,9656.0,"POLYGON ((7.07124 46.20101, 7.07111 46.20120, ...",0.836785
2,5403,Chessel,https://geo.ld.admin.ch/boundaries/municipalit...,498,355.0,"POLYGON ((6.90290 46.36155, 6.90323 46.36112, ...",1.402817
3,5404,Corbeyrier,https://geo.ld.admin.ch/boundaries/municipalit...,439,2200.0,"POLYGON ((6.98761 46.34610, 6.98753 46.34626, ...",0.199545
4,5405,Gryon,https://geo.ld.admin.ch/boundaries/municipalit...,1374,1522.0,"POLYGON ((7.15139 46.30272, 7.15028 46.30253, ...",0.902760


In [ ]:
df["html"] = "<a href='" + df["version"] + "' target='_blank'>" + df["name"] + "</a>"

geo_json = df.to_json()

colorscale = branca.colormap.linear.YlOrRd_09.scale(0, 10)

m = folium.Map(location=[47, 7], tiles='openstreetmap', zoom_start=10)

folium.GeoJson(geo_json, 
               name="geojson",
               style_function = lambda feature: {
                    "fillOpacity": 0.7,
                    "line_opacity": 0.5,
                    "weight": 1,
                    "fillColor": colorscale(feature["properties"]["density"])
               },
               highlight_function=lambda feature: {
                     "fillColor": colorscale(feature["properties"]["density"]),
                     "fillOpacity": 0.2
               },
               tooltip=folium.features.GeoJsonTooltip(
                     fields=["name", "population", "area", "density"],
                     style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"),
                     sticky=False
               ),
               popup=folium.features.GeoJsonPopup(
                    fields=["html", "population"],
                    aliases=["Gemeinde", "Einwohner"],
                    localize=False,
                    labels=True
               )
).add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds([sw, ne])
m